# Representation Probing — MaxEnt Unlearning

Standard unlearning evaluation only checks **outputs** (perplexity, relearning speed).
A model can pass all those tests while still encoding forgotten knowledge in its
intermediate representations — suppressing it at the final layer without truly erasing it.

### Core question
Can a simple **linear probe** distinguish forget-set examples from holdout examples
using only the model's hidden activations at layer $\ell$?

$$\text{probe accuracy at layer } \ell \approx 50\% \implies \text{layer } \ell \text{ no longer encodes membership signal}$$

### Models under test
All checkpoints are loaded from `../models/maxent/` (saved by `experiments-maxent-distillation.ipynb`):

| Model | Path | Expected signal |
|---|---|---|
| **Vanilla** | `EleutherAI/pythia-160m` | ≈ 50% — never saw TOFU |
| **Fine-tuned** | `pythia-160m-finetuned-tofu` | >> 50% — TOFU encoded |
| **MaxEnt Unlearned** | `pythia-160m-unlearned` | May stay >> 50% (latent trace) |
| **MaxEnt + Distilled** | `pythia-160m-distilled` | Should drop toward vanilla |

### Feature extraction
For each sample $x$ and layer $\ell$, we **mean-pool** hidden states over the token dimension:
$$\mathbf{h}_\ell(x) = \frac{1}{T}\sum_{t=1}^{T} \mathbf{H}_\ell^{(t)}(x) \in \mathbb{R}^{d}$$

### Controls
1. **Vanilla base model** — probe on a model that never saw TOFU. If it's already high, the probe detects prompt style, not knowledge.
2. **Shuffled labels** — probe accuracy should drop to 50%.
3. **Identical prompt template** — forget and holdout sets use the same `Question:/Answer:` format.
4. **Held-out non-members** — `holdout10` authors were never in any training split.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from tqdm import tqdm

# Make sure repo root is on the path
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

BASE_MODEL = 'EleutherAI/pythia-160m'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

TOFU_SPLIT        = 'forget10'   # split the MaxEnt model was trained to forget
NONMEMBER_SPLIT   = 'retain90'   # used as non-members (different fictional authors)
NONMEMBER_SAMPLES = 200          # cap non-member count to match member count
MAX_LENGTH        = 256
PROBE_CV_FOLDS    = 5
PROBE_C           = 1.0          # logistic regression regularisation strength

# Paths to saved checkpoints from experiments-maxent-distillation.ipynb
MODEL_DIR         = '../models/maxent'
PATHS = {
    'Vanilla':           BASE_MODEL,
    'Fine-tuned':        f'{MODEL_DIR}/pythia-160m-finetuned-tofu',
    'MaxEnt Unlearned':  f'{MODEL_DIR}/pythia-160m-unlearned',
    'MaxEnt + Distilled': f'{MODEL_DIR}/pythia-160m-distilled',
}

print(f'TOFU split (members)     : {TOFU_SPLIT}')
print(f'Non-member split         : {NONMEMBER_SPLIT}')
print(f'Non-member samples       : {NONMEMBER_SAMPLES}')
print(f'Probe CV folds           : {PROBE_CV_FOLDS}')
print()
for name, path in PATHS.items():
    exists = os.path.isdir(path) if path != BASE_MODEL else True
    print(f'  {name:<24} {path}  {"✓" if exists else "✗ MISSING"}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def tofu_texts(split):
    ds = load_dataset('locuslab/TOFU', split)['train']
    return [f'Question: {q}\nAnswer: {a}{tokenizer.eos_token}'
            for q, a in zip(ds['question'], ds['answer'])]


def extract_hidden_states(model, texts, max_length=MAX_LENGTH, batch_size=8):
    """Mean-pool hidden states at every layer for a list of texts.

    Returns: dict  layer_idx → np.array of shape (n_samples, hidden_dim)
    """
    model.eval()
    n_layers   = model.config.num_hidden_layers          # 12 for pythia-160m
    layer_vecs = {i: [] for i in range(n_layers + 1)}   # 0 = embedding

    with torch.no_grad():
        for text in tqdm(texts, desc='  hidden states', leave=False):
            enc = tokenizer(text, return_tensors='pt',
                            truncation=True, max_length=max_length).to(device)
            out = model(**enc, output_hidden_states=True)
            # hidden_states: tuple of (n_layers+1) tensors, each (1, T, d)
            for layer_idx, h in enumerate(out.hidden_states):
                layer_vecs[layer_idx].append(h[0].mean(dim=0).cpu().float().numpy())

    return {k: np.array(v) for k, v in layer_vecs.items()}


def train_probe(X_member, X_nonmember, n_splits=PROBE_CV_FOLDS, C=PROBE_C):
    """Logistic regression probe with stratified k-fold CV.
    Returns dict with mean/std accuracy and AUC."""
    X = np.concatenate([X_member, X_nonmember])
    y = np.concatenate([np.ones(len(X_member)), np.zeros(len(X_nonmember))])
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, C=C, solver='lbfgs')),
    ])
    accs = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    aucs = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')
    return {'acc_mean': accs.mean(), 'acc_std': accs.std(),
            'auc_mean': aucs.mean(), 'auc_std': aucs.std()}


def train_probe_shuffled(X_member, X_nonmember, n_splits=PROBE_CV_FOLDS, C=PROBE_C):
    """Same probe but with shuffled labels — should converge to ~0.50 (sanity check)."""
    X = np.concatenate([X_member, X_nonmember])
    y = np.concatenate([np.ones(len(X_member)), np.zeros(len(X_nonmember))])
    rng = np.random.default_rng(seed=0)
    y_shuf = rng.permutation(y)
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=2000, C=C, solver='lbfgs')),
    ])
    accs = cross_val_score(pipe, X, y_shuf, cv=cv, scoring='accuracy')
    return accs.mean()


print('Helpers defined.')

In [ ]:
# ── Load Models ───────────────────────────────────────────────────────────────

models = {}
for name, path in PATHS.items():
    print(f'Loading {name} from {path} ...')
    m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32).to(device)
    m.eval()
    models[name] = m

N_LAYERS = models['Vanilla'].config.num_hidden_layers  # 12 for pythia-160m
print(f'\nAll models loaded.  Transformer layers: {N_LAYERS}')

In [ ]:
# ── Prepare Member / Non-Member Texts ─────────────────────────────────────────
# member_texts    = forget10 QA pairs (what the MaxEnt model was trained to forget)
# nonmember_texts = retain90 QA pairs (different fictional authors — same QA format)
#
# Control: both sets use the identical Question:/Answer: template so the probe
# cannot exploit formatting differences.

member_texts = tofu_texts(TOFU_SPLIT)
print(f'Member texts    : {len(member_texts)} (TOFU {TOFU_SPLIT})')

all_nonmember = tofu_texts(NONMEMBER_SPLIT)
rng           = np.random.default_rng(seed=42)
nonmember_texts = list(rng.choice(all_nonmember,
                                   size=min(NONMEMBER_SAMPLES, len(all_nonmember)),
                                   replace=False))
print(f'Non-member texts: {len(nonmember_texts)} (sampled from TOFU {NONMEMBER_SPLIT})')
print(f'Class balance   : {len(member_texts)} members vs {len(nonmember_texts)} non-members')

In [ ]:
# ── Extract Hidden States for All Models ──────────────────────────────────────
# For each model, run the full text set through and collect mean-pooled
# hidden states at every layer.  Most compute-intensive cell.

hidden_states = {}  # model_name → {'member': {layer: arr}, 'nonmember': {layer: arr}}

for name, model in models.items():
    print(f'\n── {name} ──────────────────────────────────────────')
    hidden_states[name] = {
        'member':    extract_hidden_states(model, member_texts),
        'nonmember': extract_hidden_states(model, nonmember_texts),
    }
    sample_shape = hidden_states[name]['member'][0].shape
    print(f'  Done. Feature shape per layer: {sample_shape}')

print('\nAll hidden states extracted.')

In [ ]:
# ── Train Probes — Layer × Model Grid ────────────────────────────────────────
# For each (model, layer) pair, train a logistic regression probe.
# probe_results[model_name][layer_idx] = {acc_mean, acc_std, auc_mean, auc_std}

probe_results = {name: {} for name in models}
layers = list(range(N_LAYERS + 1))

for name in models:
    print(f'Probing {name} ...')
    for layer_idx in layers:
        X_m = hidden_states[name]['member'][layer_idx]
        X_n = hidden_states[name]['nonmember'][layer_idx]
        probe_results[name][layer_idx] = train_probe(X_m, X_n)
    best_layer = max(layers, key=lambda l: probe_results[name][l]['acc_mean'])
    best_acc   = probe_results[name][best_layer]['acc_mean']
    print(f'  best layer={best_layer}  acc={best_acc:.3f}')

print('\nAll probes trained.')

In [ ]:
# ── Control: Shuffled Labels ──────────────────────────────────────────────────
# Probe should drop to ≈ 0.50.  Uses the fine-tuned model's best layer.

ft_best_layer = max(layers, key=lambda l: probe_results['Fine-tuned'][l]['acc_mean'])
X_m_ft = hidden_states['Fine-tuned']['member'][ft_best_layer]
X_n_ft = hidden_states['Fine-tuned']['nonmember'][ft_best_layer]
shuffled_acc = train_probe_shuffled(X_m_ft, X_n_ft)
print(f'Shuffled-labels control (Fine-tuned, layer {ft_best_layer}): acc = {shuffled_acc:.3f}')
print(f'Expected ≈ 0.50 — confirms probe is not exploiting spurious features.')

In [ ]:
# ── Plot 1: Probe Accuracy & AUC vs Layer Depth ───────────────────────────────

MODEL_NAMES = list(models.keys())
COLORS = {
    'Vanilla':            '#7f7f7f',
    'Fine-tuned':         '#d62728',
    'MaxEnt Unlearned':   '#ff7f0e',
    'MaxEnt + Distilled': '#2ca02c',
}
STYLES = {'Vanilla': '--', 'Fine-tuned': '-', 'MaxEnt Unlearned': '-', 'MaxEnt + Distilled': '-'}

layer_labels = ['emb'] + [str(i) for i in range(1, N_LAYERS + 1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, ylabel, ax in [
    ('acc_mean', 'Probe Accuracy (5-fold CV)', axes[0]),
    ('auc_mean', 'Probe AUC (5-fold CV)',      axes[1]),
]:
    std_key = metric.replace('mean', 'std')
    ax.axhline(0.5, color='black', linestyle=':', linewidth=1, label='Random (0.50)')
    for name in MODEL_NAMES:
        vals = [probe_results[name][l][metric] for l in layers]
        errs = [probe_results[name][l][std_key] for l in layers]
        ax.plot(layers, vals, color=COLORS[name], linestyle=STYLES[name],
                linewidth=2, marker='o', markersize=4, label=name)
        ax.fill_between(layers,
                        [v - e for v, e in zip(vals, errs)],
                        [v + e for v, e in zip(vals, errs)],
                        color=COLORS[name], alpha=0.12)
    ax.set_xticks(layers)
    ax.set_xticklabels(layer_labels, fontsize=8)
    ax.set_xlabel('Layer  (0 = embedding,  1–12 = transformer layers)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(0.35, 1.05)
    ax.legend(fontsize=9)
    ax.grid(linestyle=':', alpha=0.4)

fig.suptitle(
    f'Linear Probe on Hidden States — MaxEnt Unlearning — TOFU {TOFU_SPLIT}\n'
    f'Accuracy ≈ 0.50 = knowledge erased from representations',
    fontsize=12,
)
plt.tight_layout()
plt.savefig('probe_maxent_layer_accuracy.png', dpi=300, bbox_inches='tight')
print('Saved: probe_maxent_layer_accuracy.png')
plt.show()

In [ ]:
# ── Plot 2: Probe Accuracy Heatmap (Models × Layers) ─────────────────────────

acc_matrix = np.array([
    [probe_results[name][l]['acc_mean'] for l in layers]
    for name in MODEL_NAMES
])

fig, ax = plt.subplots(figsize=(13, 3.5))
im = ax.imshow(acc_matrix, aspect='auto', cmap='RdYlGn',
               vmin=0.45, vmax=1.0, interpolation='nearest')

ax.set_xticks(layers)
ax.set_xticklabels(layer_labels, fontsize=9)
ax.set_yticks(range(len(MODEL_NAMES)))
ax.set_yticklabels(MODEL_NAMES, fontsize=10)
ax.set_xlabel('Layer  (0 = embedding,  1–12 = transformer layers)', fontsize=10)

for i, name in enumerate(MODEL_NAMES):
    for j, l in enumerate(layers):
        val = acc_matrix[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color='black' if 0.55 < val < 0.90 else 'white')

cbar = fig.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
cbar.set_label('Probe accuracy', fontsize=9)
ax.set_title(
    f'Probe Accuracy Heatmap — MaxEnt — TOFU {TOFU_SPLIT}  '
    f'(green ≈ 0.5 = erased,  red = knowledge still in representations)',
    fontsize=11,
)
plt.tight_layout()
plt.savefig('probe_maxent_heatmap.png', dpi=300, bbox_inches='tight')
print('Saved: probe_maxent_heatmap.png')
plt.show()

In [ ]:
# ── Plot 3: Accuracy Drop from Fine-tuned ────────────────────────────────────
# Δ acc = probe_acc(Fine-tuned) - probe_acc(model)
# Larger = more knowledge erased in representation space.

ft_accs = np.array([probe_results['Fine-tuned'][l]['acc_mean'] for l in layers])

fig, ax = plt.subplots(figsize=(10, 4))
for name, color in [('Vanilla', '#7f7f7f'),
                     ('MaxEnt Unlearned', '#ff7f0e'),
                     ('MaxEnt + Distilled', '#2ca02c')]:
    accs  = np.array([probe_results[name][l]['acc_mean'] for l in layers])
    delta = ft_accs - accs
    ax.plot(layers, delta, color=color, linewidth=2,
            marker='o', markersize=4, label=name)

ax.axhline(0, color='#d62728', linestyle='--', linewidth=1.2, label='Fine-tuned (Δ = 0)')
ax.set_xticks(layers)
ax.set_xticklabels(layer_labels, fontsize=8)
ax.set_xlabel('Layer  (0 = embedding,  1–12 = transformer layers)', fontsize=10)
ax.set_ylabel('Accuracy drop vs Fine-tuned  (↑ = more erased)', fontsize=10)
ax.set_title(
    f'Representation-Space Erasure — MaxEnt — TOFU {TOFU_SPLIT}\n'
    f'How much each step reduces membership signal per layer',
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('probe_maxent_erasure.png', dpi=300, bbox_inches='tight')
print('Saved: probe_maxent_erasure.png')
plt.show()

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────

ft_best_acc = max(probe_results['Fine-tuned'][l]['acc_mean'] for l in layers)

def interpret(acc):
    if acc < 0.55: return 'erased'
    if acc < 0.70: return 'partially erased'
    if acc < 0.85: return 'latent trace present'
    return 'strongly encoded'

print('─' * 84)
print(f"{'Model':<24}  {'Best Layer':>10}  {'Acc':>6}  {'AUC':>6}  "
      f"{'Δ Acc vs FT':>12}  Interpretation")
print('─' * 84)
for name in MODEL_NAMES:
    bl   = max(layers, key=lambda l: probe_results[name][l]['acc_mean'])
    acc  = probe_results[name][bl]['acc_mean']
    auc  = probe_results[name][bl]['auc_mean']
    delta = ft_best_acc - acc
    print(f"{name:<24}  {bl:>10}  {acc:>6.3f}  {auc:>6.3f}  {delta:>+12.3f}  {interpret(acc)}")
print('─' * 84)
print()
print('Interpretation:')
print('  Acc < 0.55 across all layers      → no membership signal (ideal)')
print('  Drop: Unlearned ≈ 0, Distilled large → distillation erases latent traces')
print('  Drop: Unlearned large, Distilled ≈ same → MaxEnt already erases representations')